In [4]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd


In [5]:
# Check what sheets are available in the Excel file
xl_file = pd.ExcelFile("../data/SPFmicrodata.xlsx")
print("Available indicators:", xl_file.sheet_names)

# Load only the NGDP sheet
df = pd.read_excel("../data/SPFmicrodata.xlsx", sheet_name="NGDP")
print(f"\nLoaded NGDP sheet: {df.shape[0]} rows × {df.shape[1]} columns")
print("\nColumn names:", df.columns.tolist())

Available indicators: ['NGDP', 'PGDP', 'CPROF', 'UNEMP', 'EMP', 'INDPROD', 'HOUSING', 'TBILL', 'BOND', 'BAABOND', 'TBOND', 'RGDP', 'RCONSUM', 'RNRESIN', 'RRESINV', 'RFEDGOV', 'RSLGOV', 'RCBI', 'REXPORT', 'CPI5YR', 'PCE5YR', 'CPI10', 'PCE10', 'RGDP10', 'PROD10', 'STOCK10', 'BOND10', 'BILL10', 'PRGDP', 'PRPGDP', 'PRUNEMP', 'PRCCPI', 'PRCPCE', 'RECESS', 'CPI', 'CORECPI', 'PCE', 'COREPCE', 'UBAR', 'SPR_TBOND_TBILL', 'SPR_BAA_AAA', 'SPR_BAA_TBOND', 'SPR_AAA_TBOND', 'CPIF5', 'PCEF5', 'RR1_TBILL_PGDP', 'RR2_TBILL_PGDP', 'RR3_TBILL_PGDP', 'RR1_TBILL_CPI', 'RR2_TBILL_CPI', 'RR3_TBILL_CPI', 'RR1_TBILL_CCPI', 'RR2_TBILL_CCPI', 'RR3_TBILL_CCPI', 'RR1_TBILL_PCE', 'RR2_TBILL_PCE', 'RR3_TBILL_PCE', 'RR1_TBILL_CPCE', 'RR2_TBILL_CPCE', 'RR3_TBILL_CPCE']

Loaded NGDP sheet: 9145 rows × 12 columns

Column names: ['YEAR', 'QUARTER', 'ID', 'INDUSTRY', 'NGDP1', 'NGDP2', 'NGDP3', 'NGDP4', 'NGDP5', 'NGDP6', 'NGDPA', 'NGDPB']


/Users/Parimah/anaconda3/lib/python3.11/site-packages/openpyxl/worksheet/header_footer.py:48: UserWarning: Cannot parse header or footer so it will be ignored
  warn("""Cannot parse header or footer so it will be ignored""")


In [8]:
df.head()

,YEAR,QUARTER,ID,INDUSTRY,NGDP1,NGDP2,NGDP3,NGDP4,NGDP5,NGDP6,NGDPA,NGDPB
0,1968,4,1,NaN,871.0,884.0,895.0,907.0,920.0,938.0,NaN,NaN
1,1968,4,2,NaN,871.0,891.0,910.0,929.0,958.0,973.0,NaN,NaN
2,1968,4,3,NaN,871.0,883.0,894.0,906.0,924.0,944.0,NaN,NaN
3,1968,4,4,NaN,871.0,885.0,891.0,902.0,919.0,937.0,NaN,NaN
4,1968,4,5,NaN,871.0,895.0,913.0,935.0,940.0,970.0,NaN,NaN


In [ ]:
print(" Number of unique IDs:", df['ID'].nunique())
print(" Number of unique years:", df['YEAR'].nunique())
print(" Number of na values in NGDP6:", df['NGDP6'].isna().sum())

 Number of unique IDs: 462
 Number of unique years: 58
 Number of na values in NGDP6: 974
Number of na values in NGDP6_annual: 322


Forecasts for the quarterly and annual level of nominal GDP. Seasonally
adjusted, annual rate, billions $. Prior to 1992, these are forecasts for
nominal GNP. Annual forecasts are for the annual average of the quarterly
levels.

First survey to include this variable: 1968:Q4


https://www.philadelphiafed.org/-/media/FRBP/Assets/Surveys-And-Data/survey-of-professional-forecasters/spf-documentation.pdf?sc_lang=en&hash=8408A4F1BF351A3C268B40F6BC7B95AA


Actual Nominal GDP Data by Quarters 

https://fred.stlouisfed.org/series/GDP

In [43]:
# Check what sheets are available in the Excel file
NGDP_xl_file = pd.ExcelFile("../data/GDP.xlsx")
print("Available indicators:", NGDP_xl_file.sheet_names)

# Load only the NGDP sheet
NGDP_act = pd.read_excel("../data/GDP.xlsx", sheet_name="Quarterly")
# Display the first few rows of the NGDP data
print(NGDP_act.head())

Available indicators: ['README', 'Quarterly']
  observation_date      GDP
0       1947-01-01  243.164
1       1947-04-01  245.968
2       1947-07-01  249.585
3       1947-10-01  259.745
4       1948-01-01  265.742


In [44]:
# Convert the observation_date colum to year and quarter
NGDP_act['YEAR'] = NGDP_act['observation_date'].dt.year
NGDP_act['QUARTER'] = NGDP_act['observation_date'].dt.month.apply(lambda x: (x-1)//3 + 1)
NGDP_act.rename(columns={'NA000334Q': 'NGDP_actual'}, inplace=True)
# only keep the data from 1968 Q3 onwards to match the forecast data
NGDP_act = NGDP_act[(NGDP_act['YEAR'] > 1968) | ((NGDP_act['YEAR'] == 1968) & (NGDP_act['QUARTER'] >= 4))]

NGDP_act.drop(columns=['observation_date'], inplace=True)

NGDP_act.head()

,GDP,YEAR,QUARTER
87,968.030,1968,4
88,993.337,1969,1
89,1009.020,1969,2
90,1029.956,1969,3
91,1038.147,1969,4


In [ ]:

# NGDP1 = t-1 (previous quarter historical)
# NGDP2 = t+0 (nowcast, current quarter)
# NGDP3 = t+1, NGDP4 = t+2, NGDP5 = t+3, NGDP6 = t+4
horizon_offsets = {
    'NGDP1': -1,
    'NGDP2':  0,
    'NGDP3':  1,
    'NGDP4':  2,
    'NGDP5':  3,
    'NGDP6':  4,
}

# Build a fast (year, quarter) -> GDP lookup
gdp_lookup = NGDP_act.set_index(['YEAR', 'QUARTER'])['GDP']

errors_df = df[['YEAR', 'QUARTER', 'ID', 'INDUSTRY'] + list(horizon_offsets.keys())].copy()

for col, offset in horizon_offsets.items():
    # Shift survey (YEAR, QUARTER) by offset to get the target quarter
    total    = (errors_df['YEAR'] - 1) * 4 + (errors_df['QUARTER'] - 1) + offset
    t_year   = (total // 4) + 1
    t_qtr    = (total % 4) + 1

    gdp_actual = np.array([gdp_lookup.get((y, q), np.nan)
                            for y, q in zip(t_year, t_qtr)])

    errors_df[f'error_{col}'] = errors_df[col].values - gdp_actual

errors_df.head(10)
